# Part 2: Autonomous Lane Detection — Hough Line Transform

**Course:** Digital Image Processing — HW3  
**Topic:** Polar Hough Line Transform from scratch

### Pipeline Overview
```
Input Image
    → Preprocessing  (Color Threshold + Gaussian Blur + ROI Mask)
    → Edge Detection (Sobel + NMS + Hysteresis)
    → Hough Voting   (Polar Hough Transform)
    → Post-processing (Lane Averaging + Temporal Smoothing + Overlay)
    → Animated GIF Output
```

### Rules Followed
- ✅ No forbidden CV functions (`cv2.Canny`, `cv2.Sobel`, `cv2.HoughLines`, etc.)
- ✅ All core algorithms implemented from scratch with NumPy
- ✅ Execution time profiled and printed for every stage

## Cell 1 — Imports

In [14]:
import os
import glob
import math
import time
import re
import numpy as np
import cv2
import imageio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import deque

## Cell 2 — Gaussian Kernel & 2D Convolution

Convolution is implemented using `stride_tricks` + `einsum` — fully vectorized with no pixel-level loops.
A sliding-window view of shape `(H, W, kH, kW)` is built without copying data, then contracted
with the kernel in a single `einsum` call.

In [15]:
def gaussian_kernel(size: int, sigma: float) -> np.ndarray:
    """Build a normalized 2D Gaussian kernel of given size and sigma."""
    k = size // 2
    kernel = np.zeros((size, size), dtype=np.float64)
    for i in range(size):
        for j in range(size):
            x, y = i - k, j - k
            kernel[i, j] = math.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel /= kernel.sum()
    return kernel


def convolve2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """2D convolution via stride_tricks + einsum. No pixel loops."""
    kH, kW = kernel.shape
    pH, pW = kH // 2, kW // 2
    padded  = np.pad(image.astype(np.float64), ((pH, pH), (pW, pW)), mode="reflect")
    H, W    = image.shape
    shape   = (H, W, kH, kW)
    strides = (padded.strides[0], padded.strides[1],
               padded.strides[0], padded.strides[1])
    windows = np.lib.stride_tricks.as_strided(padded, shape=shape, strides=strides)
    return np.einsum("ijkl,kl->ij", windows, kernel)


def gaussian_blur(gray: np.ndarray, size: int = 7, sigma: float = 1.5) -> np.ndarray:
    """Apply Gaussian smoothing using the from-scratch convolution."""
    return convolve2d(gray, gaussian_kernel(size, sigma))

## Cell 3 — Color Threshold

Isolates white and yellow lane markings using per-channel boolean masks.
All operations are vectorized NumPy array comparisons — no loops.

In [16]:
def color_threshold(bgr: np.ndarray) -> np.ndarray:
    """
    Return a binary mask keeping only white and yellow pixels.
    White  : all three channels above 145
    Yellow : red & green high, blue low
    """
    b = bgr[:, :, 0].astype(np.float64)
    g = bgr[:, :, 1].astype(np.float64)
    r = bgr[:, :, 2].astype(np.float64)
    white  = (r > 145) & (g > 145) & (b > 145)
    yellow = (r > 150) & (g > 130) & (b < 110)
    return (white | yellow).astype(np.float64)

## Cell 4 — Region of Interest

Applies a trapezoidal mask to keep only the road area and discard the sky and car hood.
The trapezoid vertices are defined as fixed fractions of the image dimensions.

In [17]:
def region_of_interest(mask: np.ndarray) -> np.ndarray:
    """
    Apply a trapezoidal ROI mask.

    The trapezoid spans from 90% of image height down to the bottom,
    centered horizontally. Pixels outside this region are zeroed out.
    """
    H, W = mask.shape
    roi  = np.zeros_like(mask)

    top_left     = (int(W * 0.49), int(H * 0.9))
    top_right    = (int(W * 0.51), int(H * 0.9))
    bottom_right = (int(W * 0.51), H - 1)
    bottom_left  = (int(W * 0.49), H - 1)

    min_y = top_left[1]
    for y in range(min_y, H):
        t_l = (y - bottom_left[1]) / (top_left[1] - bottom_left[1]) \
              if top_left[1] != bottom_left[1] else 1.0
        t_l = np.clip(t_l, 0, 1)
        x_left = int(bottom_left[0] + t_l * (top_left[0] - bottom_left[0]))

        t_r = (y - bottom_right[1]) / (top_right[1] - bottom_right[1]) \
              if top_right[1] != bottom_right[1] else 1.0
        t_r = np.clip(t_r, 0, 1)
        x_right = int(bottom_right[0] + t_r * (top_right[0] - bottom_right[0]))

        roi[y, x_left:x_right + 1] = mask[y, x_left:x_right + 1]

    return roi

## Cell 5 — Edge Detection (Sobel + NMS + Hysteresis)

Full Canny-style edge detector built from scratch:
1. **Sobel gradients** — `Gx` and `Gy` via from-scratch `convolve2d`
2. **Non-Maximum Suppression** — four gradient direction masks with `np.roll` (vectorized)
3. **Hysteresis Thresholding** — BFS from strong-edge seeds, accepting connected weak edges

In [18]:
SOBEL_X = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float64)

SOBEL_Y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]], dtype=np.float64)


def edge_detect(gray_blur: np.ndarray,
                low:  float = 0.08,
                high: float = 0.20) -> np.ndarray:
    """
    From-scratch edge detector: Sobel → NMS → Hysteresis.

    Parameters
    ----------
    low  : lower hysteresis threshold (weak edge acceptance)
    high : upper hysteresis threshold (strong edge seed)
    """
    # ── Sobel gradients ──────────────────────────────────────────────
    Gx        = convolve2d(gray_blur, SOBEL_X)
    Gy        = convolve2d(gray_blur, SOBEL_Y)
    magnitude = np.hypot(Gx, Gy)
    mag_norm  = magnitude / magnitude.max() if magnitude.max() > 0 else magnitude
    angle_deg = np.degrees(np.arctan2(Gy, Gx)) % 180

    # ── Non-Maximum Suppression (vectorized with np.roll) ─────────────
    q = np.ones_like(mag_norm)
    r = np.ones_like(mag_norm)

    mask0   = (angle_deg <  22.5) | (angle_deg >= 157.5)
    mask45  = (angle_deg >= 22.5) & (angle_deg <  67.5)
    mask90  = (angle_deg >= 67.5) & (angle_deg < 112.5)
    mask135 = (angle_deg >= 112.5) & (angle_deg < 157.5)

    q[mask0]   = mag_norm[mask0]   * (np.roll(mag_norm,  1, axis=1)[mask0]   <= mag_norm[mask0])
    r[mask0]   = mag_norm[mask0]   * (np.roll(mag_norm, -1, axis=1)[mask0]   <= mag_norm[mask0])
    q[mask45]  = mag_norm[mask45]  * (np.roll(np.roll(mag_norm, -1, 0),  1, 1)[mask45] <= mag_norm[mask45])
    r[mask45]  = mag_norm[mask45]  * (np.roll(np.roll(mag_norm,  1, 0), -1, 1)[mask45] <= mag_norm[mask45])
    q[mask90]  = mag_norm[mask90]  * (np.roll(mag_norm, -1, 0)[mask90]  <= mag_norm[mask90])
    r[mask90]  = mag_norm[mask90]  * (np.roll(mag_norm,  1, 0)[mask90]  <= mag_norm[mask90])
    q[mask135] = mag_norm[mask135] * (np.roll(np.roll(mag_norm, -1, 0), -1, 1)[mask135] <= mag_norm[mask135])
    r[mask135] = mag_norm[mask135] * (np.roll(np.roll(mag_norm,  1, 0),  1, 1)[mask135] <= mag_norm[mask135])

    nms = np.where((mag_norm >= q) & (mag_norm >= r), mag_norm, 0.0)

    # ── Hysteresis Thresholding (BFS) ─────────────────────────────────
    H, W    = nms.shape
    strong  = (nms >= high).astype(np.uint8)
    weak    = ((nms >= low) & (nms < high)).astype(np.uint8)
    result  = strong.copy()
    visited = strong.copy().astype(bool)
    queue   = deque(zip(*np.where(strong == 1)))

    while queue:
        y, x = queue.popleft()
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = y + dy, x + dx
                if 0 <= ny < H and 0 <= nx < W \
                        and not visited[ny, nx] and weak[ny, nx]:
                    result[ny, nx]  = 1
                    visited[ny, nx] = True
                    queue.append((ny, nx))

    return result.astype(np.float64)

## Cell 6 — Polar Hough Line Transform

For every edge pixel `(x, y)` and every angle `θ`, the corresponding `ρ` is:
$$\rho = x\cos\theta + y\sin\theta$$
All votes are cast simultaneously using broadcasting — shape `(n_edge_pixels, n_thetas)`.
`np.add.at` accumulates votes atomically into the `(ρ, θ)` accumulator.

In [19]:
def hough_transform(edge_img: np.ndarray,
                    n_thetas: int   = 360,
                    threshold_ratio: float = 0.25):
    """
    Polar Hough Line Transform from scratch.

    Returns
    -------
    lines       : list of (ρ, θ) tuples whose accumulator value >= threshold
    accumulator : 2D vote array of shape (n_rhos, n_thetas)
    rhos        : ρ axis values
    thetas      : θ axis values
    """
    H, W   = edge_img.shape
    diag   = math.ceil(math.sqrt(H**2 + W**2))
    rhos   = np.linspace(-diag, diag, 2 * diag)
    thetas = np.linspace(0, np.pi, n_thetas, endpoint=False)
    cos_t  = np.cos(thetas)
    sin_t  = np.sin(thetas)

    accumulator = np.zeros((len(rhos), n_thetas), dtype=np.int32)

    ys, xs      = np.where(edge_img > 0.5)
    # Vectorized: compute all (pixel × angle) ρ values at once
    rho_vals    = xs[:, None] * cos_t[None, :] + ys[:, None] * sin_t[None, :]
    rho_indices = np.round(
        (rho_vals - rhos[0]) / (rhos[1] - rhos[0])
    ).astype(np.int32)

    valid = (rho_indices >= 0) & (rho_indices < len(rhos))
    ri    = rho_indices[valid]
    ti    = np.tile(np.arange(n_thetas), (len(xs), 1))[valid]
    np.add.at(accumulator, (ri, ti), 1)

    threshold      = int(accumulator.max() * threshold_ratio)
    peak_r, peak_t = np.where(accumulator >= threshold)
    lines = [(rhos[r], thetas[t]) for r, t in zip(peak_r, peak_t)]
    return lines, accumulator, rhos, thetas

## Cell 7 — Lane Extraction & Averaging

Hough peaks are split into left/right groups by the sign of their slope.
A single representative line per side is fitted with **Least-Squares** regression,
producing two clean lane segments that span from the bottom of the image to a
fixed vanishing-point height.

In [20]:
def rho_theta_to_endpoints(rho: float, theta: float, H: int, W: int):
    """Intersect a Hough line (ρ, θ) with the image borders and return two endpoints."""
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    candidates = []

    if abs(cos_t) > 1e-6:
        x_at_y0 = (rho - 0 * sin_t) / cos_t
        x_at_yH = (rho - (H - 1) * sin_t) / cos_t
        if 0 <= x_at_y0 < W: candidates.append((x_at_y0, 0))
        if 0 <= x_at_yH < W: candidates.append((x_at_yH, H - 1))
    if abs(sin_t) > 1e-6:
        y_at_x0 = (rho - 0       * cos_t) / sin_t
        y_at_xW = (rho - (W - 1) * cos_t) / sin_t
        if 0 <= y_at_x0 < H: candidates.append((0, y_at_x0))
        if 0 <= y_at_xW < H: candidates.append((W - 1, y_at_xW))

    seen, unique = set(), []
    for p in candidates:
        key = (round(p[0], 1), round(p[1], 1))
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return (unique[0], unique[1]) if len(unique) >= 2 else None


def classify_and_average_lanes(lines, H: int, W: int,
                                y_bottom=None, y_top=None):
    """
    Split Hough lines into left/right by slope sign, then fit one
    representative line per side using Least-Squares.
    """
    if y_bottom is None: y_bottom = H - 1
    if y_top    is None: y_top    = int(H * 0.68)

    left_pts, right_pts = [], []

    for rho, theta in lines:
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        if abs(sin_t) < 1e-6:
            continue
        slope = -cos_t / sin_t
        if abs(slope) < 0.3:
            continue
        eps = rho_theta_to_endpoints(rho, theta, H, W)
        if eps is None:
            continue
        (left_pts if slope < 0 else right_pts).extend(eps)

    def pts_to_lane(pts):
        if len(pts) < 2:
            return None
        xs = np.array([p[0] for p in pts])
        ys = np.array([p[1] for p in pts])
        m, b = np.linalg.lstsq(np.vstack([xs, np.ones(len(xs))]).T,
                                ys, rcond=None)[0]
        if abs(m) < 1e-6:
            return None
        x_bot = int(np.clip((y_bottom - b) / m, 0, W - 1))
        x_top = int(np.clip((y_top    - b) / m, 0, W - 1))
        return (x_bot, y_bottom), (x_top, y_top)

    left_lane  = pts_to_lane(left_pts)
    right_lane = pts_to_lane(right_pts)

    if left_lane and right_lane:
        (lx_bot, ly_bot), (lx_top, ly_top) = left_lane
        (rx_bot, ry_bot), (rx_top, ry_top) = right_lane
        if lx_bot >= rx_bot:
            left_lane = right_lane = None
        elif lx_top >= rx_top:
            cx  = W // 2
            gap = int(W * 0.01)
            left_lane  = (lx_bot, ly_bot), (cx - gap, ly_top)
            right_lane = (rx_bot, ry_bot), (cx + gap, ry_top)

    return left_lane, right_lane

## Cell 8 — Temporal Smoothing

Prevents jitter between frames by blending the current detection toward the previous frame's lane.
- `current_weight` — how fast the lane reacts to new detections (0 = frozen, 1 = no smoothing)
- `max_step` — maximum pixel movement per frame, caps sudden jumps

In [21]:
def blend_lane(current_lane, previous_lane,
               current_weight: float = 0.30,
               max_step: int = 8):
    """
    Blend current lane detection toward the previous frame's lane.
    Falls back gracefully when either lane is missing.
    """
    if current_lane  is None: return previous_lane
    if previous_lane is None: return current_lane

    blended_points = []
    for curr_pt, prev_pt in zip(current_lane, previous_lane):
        cx, cy = curr_pt
        px, py = prev_pt
        dx = np.clip(cx - px, -max_step, max_step)
        dy = np.clip(cy - py, -max_step, max_step)
        blended_points.append((int(round(px + current_weight * dx)),
                                int(round(py + current_weight * dy))))
    return blended_points[0], blended_points[1]


def adapt_lanes_temporally(left_lane, right_lane, previous_lanes,
                           current_weight: float = 0.30,
                           max_step: int = 8):
    """Apply temporal blending to both left and right lanes."""
    prev_left, prev_right = previous_lanes
    return (blend_lane(left_lane,  prev_left,  current_weight, max_step),
            blend_lane(right_lane, prev_right, current_weight, max_step))

## Cell 9 — Lane Overlay Drawing

Draws the drivable-area polygon and lane lines on the image:
- **Polygon fill** — scan-line algorithm row by row with `cv2.addWeighted` for transparency
- **Lane lines** — rasterized with `np.linspace` for smooth pixel placement

In [22]:
def draw_lane_overlay(bgr: np.ndarray, left_lane, right_lane,
                      fill_color=(0, 200, 80), alpha: float = 0.35,
                      line_color=(0, 255, 100), line_thickness: int = 4) -> np.ndarray:
    """Draw semi-transparent drivable area polygon and lane boundary lines."""
    if left_lane is None or right_lane is None:
        return bgr

    overlay = bgr.copy()
    H, W    = bgr.shape[:2]

    (lx1, ly1), (lx2, ly2) = left_lane
    (rx1, ry1), (rx2, ry2) = right_lane

    # ── Polygon fill (scan-line) ──────────────────────────────────────
    poly  = np.array([[lx1, ly1], [lx2, ly2],
                       [rx2, ry2], [rx1, ry1]], dtype=np.int32)
    verts = poly.tolist() + [poly[0].tolist()]
    min_y = max(min(ly2, ry2), 0)
    max_y = min(max(ly1, ry1), H - 1)

    for y in range(min_y, max_y + 1):
        intersections = []
        for k in range(len(poly)):
            ax, ay = verts[k]
            bx, by = verts[k + 1]
            if ay == by:
                continue
            if min(ay, by) <= y < max(ay, by):
                t = (y - ay) / (by - ay)
                intersections.append(ax + t * (bx - ax))
        if len(intersections) >= 2:
            xl = int(np.clip(min(intersections), 0, W - 1))
            xr = int(np.clip(max(intersections), 0, W - 1))
            overlay[y, xl:xr + 1] = fill_color

    bgr[:] = cv2.addWeighted(overlay, alpha, bgr, 1 - alpha, 0)

    # ── Line drawing ──────────────────────────────────────────────────
    def draw_line_np(img, p1, p2, color, thickness):
        x1, y1 = p1; x2, y2 = p2
        length = math.hypot(x2 - x1, y2 - y1)
        if length < 1:
            return
        n    = int(length) * 2
        xs   = np.round(np.linspace(x1, x2, n)).astype(int)
        ys   = np.round(np.linspace(y1, y2, n)).astype(int)
        half = thickness // 2
        for xi, yi in zip(xs, ys):
            r0, r1 = max(yi - half, 0), min(yi + half + 1, H)
            c0, c1 = max(xi - half, 0), min(xi + half + 1, W)
            img[r0:r1, c0:c1] = color

    draw_line_np(bgr, (lx1, ly1), (lx2, ly2), line_color, line_thickness)
    draw_line_np(bgr, (rx1, ry1), (rx2, ry2), line_color, line_thickness)
    return bgr

## Cell 10 — Single Image Processor (with Timing)

Runs the complete pipeline on one frame and records execution time for every stage
using `time.perf_counter()` for high-resolution measurement.

In [23]:
def process_image(path: str, debug: bool = False,
                  previous_lanes=(None, None)):
    """
    Run the full lane detection pipeline on a single image.

    Returns (in debug mode)
    -----------------------
    result         : BGR image with lane overlay
    edges          : binary edge map
    accumulator    : Hough vote array
    rhos / thetas  : accumulator axis arrays
    current_lanes  : (left_lane, right_lane) for the next frame
    timing         : dict of stage → elapsed seconds
    """
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    H, W   = bgr.shape[:2]
    timing = {}

    # ── Stage 1 : Preprocessing ──────────────────────────────────────
    t0 = time.perf_counter()
    gray       = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float64) / 255.0
    color_mask = color_threshold(bgr)
    combined   = gray * color_mask
    blurred    = gaussian_blur(combined, size=7, sigma=1.4)
    roi        = region_of_interest(blurred)
    timing["Preprocessing"] = time.perf_counter() - t0

    # ── Stage 2 : Edge Detection ──────────────────────────────────────
    t0 = time.perf_counter()
    edges = edge_detect(roi, low=0.04, high=0.12)
    timing["Edge Detection"] = time.perf_counter() - t0

    # ── Stage 3 : Hough Voting ────────────────────────────────────────
    t0 = time.perf_counter()
    lines, accumulator, rhos, thetas = hough_transform(
        edges, n_thetas=360, threshold_ratio=0.20)
    timing["Hough Voting"] = time.perf_counter() - t0

    # ── Stage 4 : Post-processing ─────────────────────────────────────
    t0 = time.perf_counter()
    left_lane, right_lane = classify_and_average_lanes(lines, H, W)
    left_lane, right_lane = adapt_lanes_temporally(
        left_lane, right_lane, previous_lanes,
        current_weight=0.45, max_step=12)
    timing["Post-processing"] = time.perf_counter() - t0

    result = draw_lane_overlay(bgr.copy(), left_lane, right_lane)

    if debug:
        return result, edges, accumulator, rhos, thetas, \
               (left_lane, right_lane), timing
    return result

## Cell 11 — Main Pipeline Runner

Processes all frames sequentially, saves per-frame debug figures (4-panel),
builds the animated GIF, and prints a full timing summary table at the end.

In [24]:
def run_pipeline(input_dir:  str   = "images/line_detection",
                 output_gif: str   = "lane_detection.gif",
                 fps:        float = 2.0,
                 debug_dir:  str   = "outputs"):
    """Process all images, save debug figures, produce GIF, print timing table."""

    # ── Collect & sort images ─────────────────────────────────────────
    def numeric_sort_key(p):
        m = re.search(r'\d+', os.path.splitext(os.path.basename(p))[0])
        return int(m.group()) if m else float('inf')

    image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        image_paths.extend(glob.glob(os.path.join(input_dir, ext)))
    image_paths.sort(key=numeric_sort_key)

    if not image_paths:
        raise RuntimeError(f"No images found in '{input_dir}'")

    n = len(image_paths)
    print(f"Found {n} images in '{input_dir}'")
    os.makedirs(debug_dir, exist_ok=True)

    gif_frames     = []
    all_timings    = []
    previous_lanes = (None, None)

    # ── Process each frame ────────────────────────────────────────────
    for idx, path in enumerate(image_paths):
        fname = os.path.basename(path)
        print(f"  [{idx+1:02d}/{n}] {fname}", end="  ")

        result, edges, accumulator, rhos, thetas, previous_lanes, timing = \
            process_image(path, debug=True, previous_lanes=previous_lanes)

        all_timings.append(timing)
        total = sum(timing.values())
        print(f"Pre={timing['Preprocessing']:.3f}s  "
              f"Edge={timing['Edge Detection']:.3f}s  "
              f"Hough={timing['Hough Voting']:.3f}s  "
              f"Post={timing['Post-processing']:.3f}s  "
              f"Total={total:.3f}s")

        cv2.imwrite(os.path.join(debug_dir, f"result_{idx:03d}.png"), result)

        # ── 2×2 debug figure ───────────────────────────────────────────
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        axes[0, 0].imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title("Original", fontsize=11, fontweight="bold")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(edges, cmap="gray")
        axes[0, 1].set_title("Edge Map (Sobel + NMS + Hysteresis)", fontsize=11, fontweight="bold")
        axes[0, 1].axis("off")

        axes[1, 0].imshow(accumulator, aspect="auto", cmap="hot",
                       extent=[np.degrees(thetas[0]), np.degrees(thetas[-1]),
                                rhos[-1], rhos[0]])
        axes[1, 0].set_title("Hough Accumulator (ρ–θ)", fontsize=11, fontweight="bold")
        axes[1, 0].set_xlabel("θ (degrees)")
        axes[1, 0].set_ylabel("ρ (pixels)")

        axes[1, 1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        axes[1, 1].set_title("Lane Overlay", fontsize=11, fontweight="bold")
        axes[1, 1].axis("off")

        plt.suptitle(f"Frame {idx+1}/{n} — {fname}", fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.savefig(os.path.join(debug_dir, f"debug_{idx:03d}.png"),
                    dpi=90, bbox_inches="tight")
        plt.close(fig)

        gif_frames.append(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))

    # ── Build GIF ─────────────────────────────────────────────────────
    imageio.mimsave(output_gif, gif_frames, format="GIF", duration=1, loop=0)

    # ── Timing summary table ──────────────────────────────────────────
    stages = ["Preprocessing", "Edge Detection", "Hough Voting", "Post-processing"]
    sep    = "=" * 66
    print(f"\n{sep}")
    print("  Execution Time Summary  (seconds, averaged over all frames)")
    print(sep)
    print(f"  {'Stage':<22}  {'Mean (s)':>10}  {'Min (s)':>10}  {'Max (s)':>10}")
    print("-" * 66)
    for stage in stages:
        vals = [t[stage] for t in all_timings]
        print(f"  {stage:<22}  {np.mean(vals):>10.3f}  "
              f"{np.min(vals):>10.3f}  {np.max(vals):>10.3f}")
    totals = [sum(t.values()) for t in all_timings]
    print("-" * 66)
    print(f"  {'Total per frame':<22}  {np.mean(totals):>10.3f}  "
          f"{np.min(totals):>10.3f}  {np.max(totals):>10.3f}")
    print(sep)

    print(f"\n✓ GIF saved    → {output_gif}  ({n} frames @ {fps} fps)")
    print(f"✓ Debug frames → {debug_dir}/")
    return output_gif

## Cell 12 — Run

In [ ]:
if __name__ == "__main__":
    run_pipeline(
        input_dir  = "images/line_detection",
        output_gif = "lane_detection.gif",
        fps        = 2.0,
        debug_dir  = "outputs"
    )

Found 23 images in 'images/line_detection'
  [01/23] 1.png  Pre=0.268s  Edge=0.519s  Hough=0.088s  Post=0.438s  Total=1.314s
  [02/23] 2.png  Pre=0.276s  Edge=0.545s  Hough=0.090s  Post=0.463s  Total=1.374s
  [03/23] 3.png  Pre=0.292s  Edge=0.646s  Hough=0.159s  Post=0.484s  Total=1.582s
  [04/23] 4.png  Pre=0.280s  Edge=0.696s  Hough=0.100s  Post=0.495s  Total=1.571s
  [05/23] 5.png  Pre=0.307s  Edge=0.700s  Hough=0.112s  Post=0.512s  Total=1.630s
  [06/23] 6.png  Pre=0.296s  Edge=0.703s  Hough=0.093s  Post=0.427s  Total=1.520s
  [07/23] 7.png  Pre=0.248s  Edge=0.514s  Hough=0.081s  Post=0.429s  Total=1.270s
  [08/23] 8.png  Pre=0.309s  Edge=0.804s  Hough=0.413s  Post=0.540s  Total=2.067s
  [09/23] 9.png  Pre=0.265s  Edge=0.490s  Hough=0.081s  Post=0.416s  Total=1.252s
  [10/23] 10.png  Pre=0.246s  Edge=0.624s  Hough=0.102s  Post=0.551s  Total=1.524s
  [11/23] 11.png  Pre=0.268s  Edge=0.644s  Hough=0.106s  Post=0.499s  Total=1.517s
  [12/23] 12.png  Pre=0.255s  Edge=0.601s  Hough=0.10